# Conditional DDPM on MNIST
Train a time- and class-conditioned residual U-Net to predict Gaussian noise, then generate requested digits with classifier-free guidance. Images are padded to 32 x 32 and normalized to [-1, 1].

This is a cleaned version of the seminar experiment by Valentyn Skalenchuk. Run cells in order in a GPU Colab runtime or Jupyter environment with the repository dependencies installed. The default eight-minute budget is a demonstration, not the training duration behind the historical gallery. Checkpoints are not distributed.

The two embeddings have shape (batch, 256). Their sum is projected to (batch, channels) in each residual block and broadcast across the spatial dimensions. Skip features are concatenated along channels. The training objective is mean squared error between predicted and sampled noise.


In [ ]:
import os, math, time, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## Configuration
Set DDPM_CKPT_DIR to a persistent location before running if you want checkpoints to survive a Colab reset. RESUME restores weights and optimizer state; it does not restore the exact minibatch order or random state.

In [ ]:
T            = 1000          # diffusion timesteps
BATCH_SIZE   = 128
LR           = 2e-4
MAX_MINUTES  = 8.0           # training budget
P_UNCOND     = 0.1           # label-drop prob (for CFG)
NUM_CLASSES  = 10
NULL_CLASS   = NUM_CLASSES   # token used for "no label"
BASE_CH      = 64
TIME_DIM     = 256
EMA_DECAY    = 0.999
CKPT_DIR = os.environ.get("DDPM_CKPT_DIR", "./checkpoints/ddpm")
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CKPT_DIR, "ddpm_mnist_ckpt.pt")
RESUME       = True

## Data and forward diffusion

In [ ]:
tf = transforms.Compose([
    transforms.Pad(2),                       # 28 -> 32
    transforms.ToTensor(),                   # [0,1]
    transforms.Normalize((0.5,), (0.5,)),    # -> [-1,1]
])
train_ds = datasets.MNIST("./data", train=True, download=True, transform=tf)
loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, drop_last=True, pin_memory=True)
print("samples:", len(train_ds))

In [ ]:
betas              = torch.linspace(1e-4, 0.02, T)
alphas             = 1.0 - betas
alphabars          = torch.cumprod(alphas, dim=0)
alphabars_prev     = F.pad(alphabars[:-1], (1, 0), value=1.0)
sqrt_alphabars     = torch.sqrt(alphabars)
sqrt_one_minus_ab  = torch.sqrt(1.0 - alphabars)
sqrt_recip_alphas  = torch.sqrt(1.0 / alphas)
posterior_var      = betas * (1.0 - alphabars_prev) / (1.0 - alphabars)

# move to device
for _n in ["betas","sqrt_one_minus_ab","sqrt_recip_alphas",
           "posterior_var","sqrt_alphabars"]:
    globals()[_n] = globals()[_n].to(DEVICE)

def extract(a, t, shape):
    """Gather schedule values at timesteps t, broadcast to image shape."""
    out = a.gather(0, t)
    return out.reshape(t.shape[0], *((1,) * (len(shape) - 1)))

def q_sample(x0, t, noise):
    """Forward diffusion: noise x0 to timestep t."""
    return (extract(sqrt_alphabars, t, x0.shape) * x0
            + extract(sqrt_one_minus_ab, t, x0.shape) * noise)

## Conditioned residual U-Net

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        args = t[:, None].float() * freqs[None]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8):
        super().__init__()
        self.mlp   = nn.Linear(emb_dim, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.skip  = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, emb):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.mlp(F.silu(emb))[:, :, None, None]   # inject conditioning
        h = F.silu(self.norm2(self.conv2(h)))
        return h + self.skip(x)

class Downsample(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__(); self.op = nn.Conv2d(in_ch, out_ch, 4, 2, 1)
    def forward(self, x): return self.op(x)

class Upsample(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.op = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                nn.Conv2d(in_ch, out_ch, 3, padding=1))
    def forward(self, x): return self.op(x)

In [ ]:
class CondUNet(nn.Module):
    def __init__(self, base=BASE_CH, time_dim=TIME_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(base), nn.Linear(base, time_dim),
            nn.SiLU(), nn.Linear(time_dim, time_dim))
        self.class_emb = nn.Embedding(num_classes + 1, time_dim)  # +1 = null

        self.init_conv = nn.Conv2d(1, base, 3, padding=1)
        self.d1   = ResBlock(base,     base,     time_dim)
        self.dn1  = Downsample(base,   base * 2)
        self.d2   = ResBlock(base * 2, base * 2, time_dim)
        self.dn2  = Downsample(base*2, base * 4)
        self.d3   = ResBlock(base * 4, base * 4, time_dim)
        self.mid  = ResBlock(base * 4, base * 4, time_dim)
        self.up2  = Upsample(base * 4, base * 2)
        self.u2   = ResBlock(base * 4, base * 2, time_dim)   # +skip d2
        self.up1  = Upsample(base * 2, base)
        self.u1   = ResBlock(base * 2, base,     time_dim)   # +skip d1
        self.out  = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(),
                                  nn.Conv2d(base, 1, 3, padding=1))

    def forward(self, x, t, label):
        emb = self.time_mlp(t) + self.class_emb(label)
        h0 = self.init_conv(x)
        h1 = self.d1(h0, emb)
        h  = self.dn1(h1)
        h2 = self.d2(h, emb)
        h  = self.dn2(h2)
        h3 = self.d3(h, emb)
        h  = self.mid(h3, emb)
        h  = self.up2(h)
        h  = self.u2(torch.cat([h, h2], dim=1), emb)
        h  = self.up1(h)
        h  = self.u1(torch.cat([h, h1], dim=1), emb)
        return self.out(h)

In [ ]:
model = CondUNet().to(DEVICE)
total = 0
for name, p in model.named_parameters():
    total += p.numel()
    if p.dim() > 1:                       # skip the many tiny bias/norm vectors
        print(f"{name:32s} {tuple(p.shape)}")
print(f"\nTotal parameters: {total/1e6:.2f}M")

## Reverse DDPM sampler and exponential moving average

In [ ]:
@torch.no_grad()
def sample(model, labels, guidance_scale=3.0, img_size=32):
    model.eval()
    n = labels.shape[0]
    x = torch.randn(n, 1, img_size, img_size, device=DEVICE)
    null = torch.full((n,), NULL_CLASS, device=DEVICE, dtype=torch.long)
    for i in tqdm(reversed(range(T)), total=T, desc="sampling", leave=False):
        t = torch.full((n,), i, device=DEVICE, dtype=torch.long)
        eps_c = model(x, t, labels)
        eps_u = model(x, t, null)
        eps   = eps_u + guidance_scale * (eps_c - eps_u)
        coef  = extract(betas, t, x.shape) / extract(sqrt_one_minus_ab, t, x.shape)
        mean  = extract(sqrt_recip_alphas, t, x.shape) * (x - coef * eps)
        if i > 0:
            x = mean + torch.sqrt(extract(posterior_var, t, x.shape)) * torch.randn_like(x)
        else:
            x = mean
    return x.clamp(-1, 1)

In [ ]:
class EMA:
    def __init__(self, model, decay=EMA_DECAY):

        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for e, p in zip(self.ema.parameters(), model.parameters()):
            e.mul_(self.decay).add_(p, alpha=1 - self.decay)
        for eb, b in zip(self.ema.buffers(), model.buffers()):
            eb.copy_(b)

## Checkpoint loading and training

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
ema = EMA(model)
start_epoch, global_step = 0, 0

if RESUME and os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"]); ema.ema.load_state_dict(ck["ema"])
    opt.load_state_dict(ck["opt"])
    start_epoch, global_step = ck.get("epoch", 0), ck.get("step", 0)
    print(f"Resumed (epoch {start_epoch}, step {global_step})")

def save_ckpt(epoch):
    torch.save({"model": model.state_dict(), "ema": ema.ema.state_dict(),
                "opt": opt.state_dict(), "epoch": epoch, "step": global_step},
               CKPT_PATH)

In [ ]:
def train():
    global global_step
    t0 = time.time(); epoch = start_epoch
    print(f"Training up to {MAX_MINUTES} min...")
    try:
        while True:
            model.train()
            pbar = tqdm(loader, desc=f"epoch {epoch}")
            for x, y in pbar:
                x, y = x.to(DEVICE), y.to(DEVICE)
                drop = torch.rand(y.shape[0], device=DEVICE) < P_UNCOND   # CFG dropout
                y = torch.where(drop, torch.full_like(y, NULL_CLASS), y)
    
                t = torch.randint(0, T, (x.shape[0],), device=DEVICE)
                noise = torch.randn_like(x)
                x_t = q_sample(x, t, noise)
                loss = F.mse_loss(model(x_t, t, y), noise)
    
                opt.zero_grad(); loss.backward(); opt.step(); ema.update(model)
                global_step += 1
                pbar.set_postfix(loss=f"{loss.item():.4f}", min=f"{(time.time()-t0)/60:.1f}")
                if (time.time() - t0) / 60 >= MAX_MINUTES:
                    save_ckpt(epoch); print(f"\nBudget reached. Saved -> {CKPT_PATH}")
                    return
            epoch += 1; save_ckpt(epoch)
    finally:
        save_ckpt(epoch)


In [ ]:
train()


## Generate labeled digits
Sampling uses 1000 reverse steps and can take several minutes on a GPU. Each row requests one digit class. An eight-minute training run may still produce poor samples.

In [ ]:
@torch.no_grad()
def show_grid(per_digit=8, guidance_scale=3.0):
    labels = torch.arange(NUM_CLASSES, device=DEVICE).repeat_interleave(per_digit)
    imgs = (sample(ema.ema, labels, guidance_scale) + 1) / 2
    grid = make_grid(imgs, nrow=per_digit)
    plt.figure(figsize=(per_digit, NUM_CLASSES)); plt.axis("off")
    plt.title("each row = one digit (0-9)")
    plt.imshow(grid.permute(1, 2, 0).cpu()); plt.show()

show_grid(per_digit=8)

In [ ]:
@torch.no_grad()
def generate_number(s, guidance_scale=3.0):
    labels = torch.tensor([int(c) for c in s], device=DEVICE)
    imgs = (sample(ema.ema, labels, guidance_scale) + 1) / 2
    grid = make_grid(imgs, nrow=len(s))
    plt.figure(figsize=(len(s), 1.5)); plt.axis("off"); plt.title(s)
    plt.imshow(grid.permute(1, 2, 0).cpu()); plt.show()

generate_number("2026")

## Training-batch diagnostic
This is one noisy training batch, not a held-out validation metric. It cannot establish image quality or generalization.

In [ ]:


# 2. Run a verification step to evaluate the loss on a batch
model.eval()  # Put model in evaluation mode
with torch.no_grad():
    # Grab a single batch from your training loader
    x, y = next(iter(loader))
    x, y = x.to(DEVICE), y.to(DEVICE)

    # Run the exact same forward pass logic from your training loop
    t = torch.randint(0, T, (x.shape[0],), device=DEVICE)
    noise = torch.randn_like(x)
    x_t = q_sample(x, t, noise)  # Assuming q_sample is initialized

    # Predict noise and calculate loss
    pred_noise = model(x_t, t, y)
    test_loss = F.mse_loss(pred_noise, noise)

print("\n================ VERIFICATION ================")
print(f"Single training-batch noise MSE: {test_loss.item():.5f}")
print("==============================================")
